# Buddhist Texts Corpus - Text Extraction Pipeline

This notebook extracts text from downloaded Buddhist PDFs using:
- **pdfplumber** for text-based PDFs (free)
- **Google Document AI** for scanned/image PDFs (with OCR confidence tracking)

## Features
- ✅ Copyright filtering (70-year rule)
- ✅ Unified metadata management
- ✅ OCR confidence tracking
- ✅ Page classification (cover/TOC/content/index)
- ✅ Resumable extraction
- ✅ Book-level directory structure
- ✅ Quality metrics and reporting

## Output Structure
```
data/
├── extracted/
│   ├── raw_text/{category}/{book_name}/page_001.txt
│   ├── filtered_text/{category}/{book_name}/page_001.txt
│   └── book_metadata/{category}/{book_name}/extraction_info.json
└── metadata/
    └── metadata.json  # Updated with extraction info
```

## 1. Setup and Installation

In [2]:
# Install required packages
!pip install -q pdfplumber PyPDF2 pycryptodome google-cloud-documentai google-cloud-storage tqdm pandas matplotlib seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 103.0 MB/s eta 0:00:00


In [3]:
# Import libraries
import os
import json
import re
import time
import hashlib
import unicodedata
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pdfplumber
import PyPDF2
import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

# Google Cloud imports
from google.cloud import documentai_v1 as documentai
from google.cloud import storage
from google.oauth2 import service_account
from google.api_core.client_options import ClientOptions

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Mount Google Drive and Configuration

In [4]:
# Mount Google Drive
drive.mount('/content/drive')

# Project directory configuration
BASE_DIR = Path('/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/')

# Verify project directory
if BASE_DIR.exists():
    print(f"✓ Project directory found: {BASE_DIR}")
else:
    print(f"✗ Project directory not found: {BASE_DIR}")
    print("Please update BASE_DIR to match your Google Drive structure")

Mounted at /content/drive
✓ Project directory found: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent


In [5]:
# Directory paths
RAW_PDF_DIR = BASE_DIR / "data" / "raw"
EXTRACTED_DIR = BASE_DIR / "data" / "extracted"
METADATA_DIR = BASE_DIR / "data" / "metadata"

# Create extraction directories
RAW_TEXT_DIR = EXTRACTED_DIR / "raw_text"
FILTERED_TEXT_DIR = EXTRACTED_DIR / "filtered_text"
LOW_CONF_DIR = EXTRACTED_DIR / "low_confidence_pages"
DOCAI_CACHE_DIR = EXTRACTED_DIR / "document_ai_cache"
BOOK_METADATA_DIR = EXTRACTED_DIR / "book_metadata"
FULL_BOOKS_DIR = EXTRACTED_DIR / "full_books"

for dir_path in [RAW_TEXT_DIR, FILTERED_TEXT_DIR, LOW_CONF_DIR,
                 DOCAI_CACHE_DIR, BOOK_METADATA_DIR, FULL_BOOKS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Metadata file
METADATA_FILE = METADATA_DIR / "metadata.json"

print("✓ Directory structure created")
print(f"  Raw PDFs: {RAW_PDF_DIR}")
print(f"  Extracted text: {EXTRACTED_DIR}")
print(f"  Metadata: {METADATA_FILE}")

✓ Directory structure created
  Raw PDFs: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw
  Extracted text: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extracted
  Metadata: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/metadata.json


## 3. Google Document AI Configuration

In [6]:
# Google Cloud Authentication
CREDENTIALS_PATH = BASE_DIR / "auth" / "Vision OCR thematic runner.json"

if not CREDENTIALS_PATH.exists():
    print("⚠️ WARNING: Credentials file not found!")
    print(f"   Looking for: {CREDENTIALS_PATH}")
else:
    print("✓ Credentials file found")

# Google Cloud Configuration
PROJECT_ID = "thematic-runner-453210-s8"
LOCATION = "us"
PROCESSOR_ID = "11b17ff9bb09b5db"

# GCS Configuration
GCS_BUCKET_NAME = f"{PROJECT_ID}-docai-batch"
GCS_INPUT_PREFIX = "input_pdfs/"
GCS_OUTPUT_PREFIX = "output_results/"

# Processing configuration
POLL_INTERVAL = 30
MAX_POLL_ATTEMPTS = 120
RETRY_ATTEMPTS = 3

# Extraction thresholds
MIN_TEXT_LENGTH_FOR_TEXT_PDF = 100  # Minimum chars to consider text-based PDF
LOW_CONFIDENCE_THRESHOLD = 0.7      # Flag pages below this for review
HIGH_CONFIDENCE_THRESHOLD = 0.9     # High quality threshold

print("\nConfiguration:")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Processor: {PROCESSOR_ID}")
print(f"  GCS Bucket: {GCS_BUCKET_NAME}")
print(f"  Low confidence threshold: {LOW_CONFIDENCE_THRESHOLD}")
print(f"  High confidence threshold: {HIGH_CONFIDENCE_THRESHOLD}")

✓ Credentials file found

Configuration:
  Project: thematic-runner-453210-s8
  Location: us
  Processor: 11b17ff9bb09b5db
  GCS Bucket: thematic-runner-453210-s8-docai-batch
  Low confidence threshold: 0.7
  High confidence threshold: 0.9


In [7]:
# Initialize Google Cloud clients
def initialize_clients(credentials_path: Path, project_id: str, location: str):
    """Initialize Document AI and Cloud Storage clients."""
    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )

    opts = ClientOptions(api_endpoint=f"{location}-documentai.googleapis.com")
    docai_client = documentai.DocumentProcessorServiceClient(
        client_options=opts,
        credentials=credentials
    )

    storage_client = storage.Client(
        credentials=credentials,
        project=project_id
    )

    return docai_client, storage_client, credentials

try:
    docai_client, storage_client, credentials = initialize_clients(
        CREDENTIALS_PATH, PROJECT_ID, LOCATION
    )
    print("✓ Document AI client initialized")
    print("✓ Cloud Storage client initialized")
except Exception as e:
    print(f"✗ Error initializing clients: {e}")
    docai_client = None
    storage_client = None

✓ Document AI client initialized
✓ Cloud Storage client initialized


## 4. Helper Functions

In [8]:
# File I/O helpers
def load_json(filepath: Path, default=None):
    """Load JSON file or return default if not exists"""
    if filepath.exists():
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return default if default is not None else {}

def save_json(data, filepath: Path):
    """Save data to JSON file"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def sanitize_filename(filename: str) -> str:
    """Remove invalid characters from filename"""
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    if len(filename) > 200:
        filename = filename[:200]
    return filename.strip()

def calculate_sinhala_ratio(text: str) -> float:
    """Calculate ratio of Sinhala characters"""
    if not text:
        return 0.0
    sinhala_chars = sum(1 for c in text if '\u0D80' <= c <= '\u0DFF')
    return sinhala_chars / len(text)

def get_pdf_page_count(pdf_path: Path) -> int:
    """Get number of pages in PDF"""
    try:
        with open(pdf_path, 'rb') as f:
            pdf_reader = PyPDF2.PdfReader(f)
            return len(pdf_reader.pages)
    except Exception as e:
        print(f"  Warning: Could not count pages - {e}")
        return None

print("✓ Helper functions loaded")

✓ Helper functions loaded


## 5. Copyright Filtering Functions

In [9]:
def can_include_in_corpus(book_metadata: dict, current_year: int = 2025) -> Tuple[bool, str, str]:
    """
    Determine if a book can be included based on copyright law (70-year rule).

    Returns:
        (can_include: bool, reason: str, confidence: str)
    """
    category = book_metadata.get('Category', '')
    author_info = book_metadata.get('Author', {})
    author_name = author_info.get('name')
    author_dob = author_info.get('DOB')
    author_death = author_info.get('Date_of_passing')
    book_name = book_metadata.get('book_name_si', 'Unknown')

    # Criterion 1: Exclude Tripiṭaka books (hardcoded exclusion)
    if 'thripitaka' in category.lower() or 'tripitaka' in category.lower():
        return (False, "Tripiṭaka category - will be web-scraped separately (hardcoded exclusion)", "hardcoded")

    # Criterion 2: Check author death date
    if author_death:
        try:
            # Handle different date formats
            if isinstance(author_death, str):
                death_year = int(author_death.split('-')[0])
            else:
                death_year = author_death.year

            if death_year <= 1954:
                return (True, f"Public domain - author died in {death_year} (>70 years ago)", "high")
            else:
                protection_ends = death_year + 70
                years_remaining = protection_ends - current_year
                return (False, f"Copyright protected - author died in {death_year}, protection ends {protection_ends} ({years_remaining} years remaining)", "high")
        except Exception as e:
            print(f"  Warning: Error parsing death date for {book_name}: {e}")

    # Criterion 3: Author presumed alive (has DOB but no death date)
    if author_name and author_dob and not author_death:
        return (False, "Author presumed alive - copyright active", "high")

    # Criterion 4: Missing author information (conservative approach)
    if not author_name or (not author_dob and not author_death):
        return (False, "Author information missing - cannot verify public domain status (conservative exclusion)", "low")

    # Default: exclude if uncertain
    return (False, "Copyright status unclear", "low")

print("✓ Copyright filtering function loaded")

✓ Copyright filtering function loaded


## 6. Page Classification Functions

In [10]:
def classify_page(text: str, page_num: int, total_pages: int, confidence: float = 1.0) -> dict:
    """
    Classify page type using heuristics.

    Returns dict with:
        - type: 'cover', 'toc', 'content', 'index', 'blank', 'other'
        - confidence_tier: 'high', 'medium', 'low'
        - needs_review: bool
        - ocr_confidence: float
    """

    # Blank pages
    if len(text.strip()) < 50:
        page_type = 'blank'

    # Cover pages (first 3 pages, low text density)
    elif page_num <= 3 and len(text) < 500:
        page_type = 'cover'

    # Table of Contents indicators
    elif page_num <= total_pages * 0.15:  # Usually in first 15%
        toc_patterns = [
            'අන්තර්ගතය', 'අනුක්\u200dරමය', 'පටුන',  # Sinhala TOC
            'contents', 'index',  # English
            r'\.{3,}',  # Dot leaders
        ]
        if any(re.search(pattern, text.lower()) for pattern in toc_patterns):
            if text.count('\n') > 20 and len(text) < 2000:
                page_type = 'toc'
            else:
                page_type = 'content'  # Looks like TOC but might be content
        else:
            page_type = 'content'

    # Index pages (usually at end, lots of page numbers)
    elif page_num > total_pages * 0.9:
        digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
        if digit_ratio > 0.15:
            page_type = 'index'
        else:
            page_type = 'content'

    # Content pages (high Sinhala ratio, sufficient length)
    else:
        sinhala_ratio = calculate_sinhala_ratio(text)
        if sinhala_ratio > 0.3 and len(text) > 200:
            page_type = 'content'
        else:
            page_type = 'other'

    # Determine confidence tier based on OCR confidence
    if confidence >= HIGH_CONFIDENCE_THRESHOLD:
        confidence_tier = 'high'
        needs_review = False
    elif confidence >= LOW_CONFIDENCE_THRESHOLD:
        confidence_tier = 'medium'
        needs_review = False
    else:
        confidence_tier = 'low'
        needs_review = True

    return {
        'type': page_type,
        'confidence_tier': confidence_tier,
        'needs_review': needs_review,
        'ocr_confidence': confidence,
        'sinhala_ratio': calculate_sinhala_ratio(text),
        'char_count': len(text)
    }

print("✓ Page classification function loaded")

✓ Page classification function loaded


## 7. Text Extraction Functions

In [11]:
def extract_page_with_pdfplumber(pdf_path: Path, page_num: int) -> Optional[str]:
    """
    Extract text from a page using pdfplumber.
    Returns None if extraction fails or yields insufficient text.
    """
    try:
        with pdfplumber.open(pdf_path) as pdf:
            if page_num <= len(pdf.pages):
                page = pdf.pages[page_num - 1]  # pdfplumber uses 0-indexing
                text = page.extract_text()

                # Validate extraction
                if text and len(text.strip()) >= MIN_TEXT_LENGTH_FOR_TEXT_PDF:
                    return text
        return None
    except Exception as e:
        return None


def extract_page_with_document_ai(pdf_path: Path, page_num: int,
                                    docai_client, storage_client) -> Optional[dict]:
    """
    Extract text from a single page using Google Document AI.

    Returns dict with:
        - text: extracted text
        - confidence: average confidence score
        - blocks: list of text blocks with confidence
        - words: list of words with confidence
    """
    try:
        # Extract single page to temporary PDF
        temp_pdf_path = Path(f"/tmp/temp_page_{page_num}.pdf")

        # Create single-page PDF
        with open(pdf_path, 'rb') as infile:
            reader = PyPDF2.PdfReader(infile)
            writer = PyPDF2.PdfWriter()
            writer.add_page(reader.pages[page_num - 1])

            with open(temp_pdf_path, 'wb') as outfile:
                writer.write(outfile)

        # Upload to GCS
        bucket = storage_client.bucket(GCS_BUCKET_NAME)
        blob_name = f"{GCS_INPUT_PREFIX}{pdf_path.stem}_page_{page_num}.pdf"
        blob = bucket.blob(blob_name)
        blob.upload_from_filename(str(temp_pdf_path))
        gcs_uri = f"gs://{GCS_BUCKET_NAME}/{blob_name}"

        # Process with Document AI
        processor_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/processors/{PROCESSOR_ID}"

        # Read file content
        with open(temp_pdf_path, 'rb') as f:
            file_content = f.read()

        # Create request
        raw_document = documentai.RawDocument(
            content=file_content,
            mime_type='application/pdf'
        )

        request = documentai.ProcessRequest(
            name=processor_name,
            raw_document=raw_document
        )

        # Process document
        result = docai_client.process_document(request=request)
        document = result.document

        # Extract text and confidence
        full_text = document.text

        word_confidences = []
        blocks_data = []
        words_data = []

        for page in document.pages:
            # Extract blocks
            for block in page.blocks:
                block_text = get_text_from_layout(block.layout, full_text)
                blocks_data.append({
                    'text': block_text,
                    'confidence': block.layout.confidence
                })

            # Extract words
            for token in page.tokens:
                word_text = get_text_from_layout(token.layout, full_text)
                word_conf = token.layout.confidence
                words_data.append({
                    'word': word_text,
                    'confidence': word_conf
                })
                word_confidences.append(word_conf)

        avg_confidence = sum(word_confidences) / len(word_confidences) if word_confidences else 0

        # Cleanup
        temp_pdf_path.unlink(missing_ok=True)
        blob.delete()

        return {
            'text': full_text,
            'confidence': avg_confidence,
            'blocks': blocks_data,
            'words': words_data,
            'method': 'document_ai'
        }

    except Exception as e:
        print(f"    Document AI error: {e}")
        return None


def get_text_from_layout(layout, full_text: str) -> str:
    """Extract text from Document AI layout object"""
    text = ""
    for segment in layout.text_anchor.text_segments:
        start_index = segment.start_index if hasattr(segment, 'start_index') else 0
        end_index = segment.end_index
        text += full_text[start_index:end_index]
    return text


def extract_page(pdf_path: Path, page_num: int,
                 docai_client=None, storage_client=None) -> dict:
    """
    Extract text from a page using best available method.

    Returns dict with:
        - text: extracted text
        - confidence: 1.0 for text-based, OCR confidence for scanned
        - method: 'pdfplumber' or 'document_ai'
        - metadata: additional info
    """

    # Try pdfplumber first (free and fast)
    text = extract_page_with_pdfplumber(pdf_path, page_num)

    if text:
        return {
            'text': text,
            'confidence': 1.0,
            'method': 'pdfplumber',
            'metadata': {
                'char_count': len(text),
                'sinhala_ratio': calculate_sinhala_ratio(text)
            }
        }

    # Fall back to Document AI (for scanned/image PDFs)
    if docai_client and storage_client:
        print(f"    Using Document AI (scanned page)...")
        result = extract_page_with_document_ai(pdf_path, page_num, docai_client, storage_client)

        if result:
            return result

    # Extraction failed
    return {
        'text': '',
        'confidence': 0.0,
        'method': 'failed',
        'metadata': {'error': 'All extraction methods failed'}
    }

print("✓ Text extraction functions loaded")

✓ Text extraction functions loaded


## 8. Book Processing Functions

In [12]:
def create_book_directories(category: str, book_name: str) -> dict:
    """Create all necessary directories for a book's extraction"""
    safe_book_name = sanitize_filename(book_name)

    directories = {
        'raw_text': RAW_TEXT_DIR / category / safe_book_name,
        'filtered_text': FILTERED_TEXT_DIR / category / safe_book_name,
        'low_confidence': LOW_CONF_DIR / category / safe_book_name,
        'docai_cache': DOCAI_CACHE_DIR / category / safe_book_name,
        'book_metadata': BOOK_METADATA_DIR / category / safe_book_name,
    }

    for dir_path in directories.values():
        dir_path.mkdir(parents=True, exist_ok=True)

    return directories


def save_page_text(text: str, confidence_data: dict, directories: dict,
                   page_num: int, page_type: str = 'raw'):
    """Save page text and confidence data"""
    page_filename = f"page_{page_num:03d}.txt"
    confidence_filename = f"page_{page_num:03d}_confidence.json"

    # Determine directory
    if page_type == 'raw':
        base_dir = directories['raw_text']
    elif page_type == 'filtered':
        base_dir = directories['filtered_text']
    elif page_type == 'low_confidence':
        base_dir = directories['low_confidence']
    else:
        return None, None

    text_path = base_dir / page_filename
    confidence_path = base_dir / confidence_filename

    # Save text
    with open(text_path, 'w', encoding='utf-8') as f:
        f.write(text)

    # Save confidence
    save_json(confidence_data, confidence_path)

    return text_path, confidence_path


def check_extraction_status(directories: dict, total_pages: int) -> dict:
    """Check if book extraction is complete"""
    raw_text_dir = directories['raw_text']

    if not raw_text_dir.exists():
        return {
            'status': 'not_started',
            'pages_extracted': 0,
            'pages_remaining': total_pages,
            'completion_percentage': 0
        }

    page_files = list(raw_text_dir.glob('page_*.txt'))
    pages_extracted = len(page_files)

    if pages_extracted == 0:
        status = 'not_started'
    elif pages_extracted < total_pages:
        status = 'in_progress'
    else:
        status = 'complete'

    last_page = 0
    if page_files:
        page_nums = [int(f.stem.split('_')[1]) for f in page_files]
        last_page = max(page_nums)

    return {
        'status': status,
        'pages_extracted': pages_extracted,
        'pages_remaining': total_pages - pages_extracted,
        'completion_percentage': (pages_extracted / total_pages * 100) if total_pages > 0 else 0,
        'last_page': last_page
    }

print("✓ Book processing functions loaded")

✓ Book processing functions loaded


## 9. Main Extraction Pipeline

In [13]:
def extract_book(book_metadata: dict, docai_client=None, storage_client=None) -> dict:
    """
    Extract all pages from a book and update metadata.

    Returns updated book_metadata with extraction info.
    """

    book_name_si = book_metadata.get('book_name_si', 'Unknown')
    book_name_en = book_metadata.get('book_name_en', '')
    category = book_metadata.get('Category', 'unknown')
    pdf_path = Path(book_metadata['local_path'])

    # Display book name
    display_name = f"{book_name_si}"
    if book_name_en:
        display_name += f" ({book_name_en})"
    print(f"\n{'='*80}")
    print(f"EXTRACTING: {display_name}")
    print(f"Category: {category}")
    print('='*80)

    # Get page count
    total_pages = get_pdf_page_count(pdf_path)
    if not total_pages:
        print("  ✗ Could not determine page count")
        return book_metadata

    print(f"Total pages: {total_pages}")

    # Create directories
    directories = create_book_directories(category, book_name_si)

    # Check extraction status
    status = check_extraction_status(directories, total_pages)

    if status['status'] == 'complete':
        print("  ✓ Already fully extracted")
        return book_metadata

    if status['status'] == 'in_progress':
        print(f"  ⟳ Resuming from page {status['last_page'] + 1}")
        start_page = status['last_page'] + 1
    else:
        start_page = 1

    # Initialize extraction tracking
    extraction_start_time = time.time()
    page_classifications = []
    ocr_confidences = []
    method_counts = {'pdfplumber': 0, 'document_ai': 0, 'failed': 0}
    low_confidence_pages = []

    # Extract pages
    for page_num in tqdm(range(start_page, total_pages + 1),
                         desc="Extracting pages",
                         initial=status['pages_extracted']):

        # Extract text
        result = extract_page(pdf_path, page_num, docai_client, storage_client)

        # Track method
        method_counts[result['method']] += 1

        # Classify page
        classification = classify_page(
            result['text'],
            page_num,
            total_pages,
            result['confidence']
        )

        page_classifications.append({
            'page_num': page_num,
            'classification': classification,
            'method': result['method']
        })

        # Track confidence
        if result['method'] == 'document_ai':
            ocr_confidences.append(result['confidence'])
            if result['confidence'] < LOW_CONFIDENCE_THRESHOLD:
                low_confidence_pages.append(page_num)

        # Save to raw_text (all pages)
        confidence_data = {
            'confidence': result['confidence'],
            'method': result['method'],
            'classification': classification,
            'metadata': result.get('metadata', {})
        }

        save_page_text(result['text'], confidence_data, directories, page_num, 'raw')

        # Save to filtered_text (content pages only)
        if classification['type'] == 'content':
            save_page_text(result['text'], confidence_data, directories, page_num, 'filtered')

        # Save to low_confidence (if needed)
        if classification['needs_review']:
            save_page_text(result['text'], confidence_data, directories, page_num, 'low_confidence')

    extraction_duration = time.time() - extraction_start_time

    # Calculate statistics
    content_pages = sum(1 for p in page_classifications if p['classification']['type'] == 'content')

    # Build extraction metadata
    extraction_info = {
        'status': 'complete',
        'extraction_timestamp': datetime.now().isoformat(),
        'extraction_duration_seconds': round(extraction_duration, 2),

        'page_info': {
            'total_pages': total_pages,
            'text_based_pages': method_counts['pdfplumber'],
            'ocr_pages': method_counts['document_ai'],
            'failed_pages': method_counts['failed']
        },

        'methods_used': method_counts,

        'page_classification': {
            'cover': sum(1 for p in page_classifications if p['classification']['type'] == 'cover'),
            'toc': sum(1 for p in page_classifications if p['classification']['type'] == 'toc'),
            'content': content_pages,
            'index': sum(1 for p in page_classifications if p['classification']['type'] == 'index'),
            'blank': sum(1 for p in page_classifications if p['classification']['type'] == 'blank'),
            'other': sum(1 for p in page_classifications if p['classification']['type'] == 'other')
        },

        'quality_metrics': {},

        'file_paths': {
            'raw_text_dir': str(directories['raw_text'].relative_to(BASE_DIR)),
            'filtered_text_dir': str(directories['filtered_text'].relative_to(BASE_DIR)),
            'book_metadata_dir': str(directories['book_metadata'].relative_to(BASE_DIR))
        }
    }

    # Add OCR confidence metrics if applicable
    if ocr_confidences:
        extraction_info['quality_metrics']['ocr_confidence'] = {
            'avg_confidence': round(sum(ocr_confidences) / len(ocr_confidences), 3),
            'min_confidence': round(min(ocr_confidences), 3),
            'max_confidence': round(max(ocr_confidences), 3),
            'low_confidence_pages': low_confidence_pages,
            'distribution': {
                'high_0.9_to_1.0': sum(1 for c in ocr_confidences if c >= 0.9),
                'medium_0.7_to_0.9': sum(1 for c in ocr_confidences if 0.7 <= c < 0.9),
                'low_below_0.7': sum(1 for c in ocr_confidences if c < 0.7)
            }
        }

    # Save page classifications
    classifications_file = directories['book_metadata'] / 'page_classifications.json'
    save_json(page_classifications, classifications_file)

    # Save extraction info
    extraction_file = directories['book_metadata'] / 'extraction_info.json'
    save_json(extraction_info, extraction_file)

    # Update book metadata
    book_metadata['extraction'] = extraction_info

    print(f"\n✓ Extraction complete")
    print(f"  Duration: {extraction_duration:.1f}s")
    print(f"  Content pages: {content_pages}/{total_pages}")
    print(f"  Methods: {method_counts['pdfplumber']} text-based, {method_counts['document_ai']} OCR")
    if ocr_confidences:
        avg_conf = sum(ocr_confidences) / len(ocr_confidences)
        print(f"  Avg OCR confidence: {avg_conf:.3f}")

    return book_metadata

print("✓ Main extraction pipeline loaded")

✓ Main extraction pipeline loaded


## 10. Run Extraction Pipeline

In [14]:
# Load metadata
print("Loading metadata...")
metadata = load_json(METADATA_FILE, [])
print(f"✓ Loaded {len(metadata)} books")

# Filter by copyright
print("\n" + "="*80)
print("APPLYING COPYRIGHT FILTERS")
print("="*80)

included_books = []
excluded_books = []

for book in metadata:
    can_include, reason, confidence = can_include_in_corpus(book)

    # Add copyright analysis to metadata
    book['copyright_analysis'] = {
        'can_include_in_corpus': can_include,
        'reason': reason,
        'confidence': confidence,
        'analysis_date': datetime.now().isoformat()
    }

    if can_include:
        included_books.append(book)
    else:
        excluded_books.append(book)

print(f"\nCopyright Filtering Results:")
print(f"  Total books: {len(metadata)}")
print(f"  ✓ Included: {len(included_books)}")
print(f"  ✗ Excluded: {len(excluded_books)}")

# Show exclusion reasons
exclusion_reasons = {}
for book in excluded_books:
    reason = book['copyright_analysis']['reason']
    exclusion_reasons[reason] = exclusion_reasons.get(reason, 0) + 1

print(f"\nExclusion reasons:")
for reason, count in sorted(exclusion_reasons.items(), key=lambda x: x[1], reverse=True):
    print(f"  {count:3d} - {reason}")

Loading metadata...
✓ Loaded 83 books

APPLYING COPYRIGHT FILTERS

Copyright Filtering Results:
  Total books: 83
  ✓ Included: 16
  ✗ Excluded: 67

Exclusion reasons:
   18 - Author presumed alive - copyright active
   13 - Author information missing - cannot verify public domain status (conservative exclusion)
   10 - Tripiṭaka category - will be web-scraped separately (hardcoded exclusion)
    5 - Copyright protected - author died in 2016, protection ends 2086 (61 years remaining)
    4 - Copyright protected - author died in 1997, protection ends 2067 (42 years remaining)
    4 - Copyright protected - author died in 1992, protection ends 2062 (37 years remaining)
    4 - Copyright protected - author died in 2001, protection ends 2071 (46 years remaining)
    3 - Copyright protected - author died in 2018, protection ends 2088 (63 years remaining)
    2 - Copyright protected - author died in 1998, protection ends 2068 (43 years remaining)
    1 - Copyright protected - author died in 1

In [16]:
# Extract text from included books
print("\n" + "="*80)
print("STARTING TEXT EXTRACTION")
print("="*80)
print(f"Processing {len(included_books)} books...\n")

extraction_errors = []

for idx, book in enumerate(included_books, 1):
    print(f"\n[{idx}/{len(included_books)}]")

    try:
        updated_book = extract_book(book, docai_client, storage_client)

        # Update in metadata list
        for i, b in enumerate(metadata):
            if (b['book_name_si'] == book['book_name_si'] and
                b['book_name_en'] == book['book_name_en']):
                metadata[i] = updated_book
                break

        # Save metadata periodically (every 5 books)
        if idx % 5 == 0:
            save_json(metadata, METADATA_FILE)
            print("  💾 Metadata saved")

    except Exception as e:
        print(f"  ✗ Error: {e}")
        extraction_errors.append({
            'book': book['book_name_si'],
            'error': str(e)
        })

# Final save
save_json(metadata, METADATA_FILE)
print("\n" + "="*80)
print("EXTRACTION COMPLETE")
print("="*80)
print(f"✓ Successfully processed: {len(included_books) - len(extraction_errors)}")
print(f"✗ Errors: {len(extraction_errors)}")

if extraction_errors:
    print("\nBooks with errors:")
    for err in extraction_errors:
        print(f"  - {err['book']}: {err['error']}")


STARTING TEXT EXTRACTION
Processing 16 books...


[1/16]

EXTRACTING: මෛත්‍රී බුද්ධ වංශය නොහොත් අනාගත වංශය (Maithree Buddha Wanshaya nohoth anagatha wanshaya)
Category: books-related-to-the-tipitaka
Total pages: 35
  ✓ Already fully extracted

[2/16]

EXTRACTING: මිලින්ද ප්‍රශ්නය (Milind prashna)
Category: books-related-to-the-tipitaka
Total pages: 660
  ✓ Already fully extracted

[3/16]

EXTRACTING: ප්‍රේත වස්තුව (Pretha Wasthu)
Category: books-related-to-the-tipitaka
Total pages: 435
  ✓ Already fully extracted

[4/16]

EXTRACTING: විමානවස්තුව (Vimana wasthu)
Category: books-related-to-the-tipitaka
Total pages: 445
  ✓ Already fully extracted

[5/16]

EXTRACTING: විශුද්ධි මාර්ගය – බුද්ධඝෝෂ මාහිමියන් විසිනි (Wishuddhi Margaya. – Buddhaghosha Maha Thero)
Category: books-related-to-the-tipitaka
Total pages: 1097
  ✓ Already fully extracted
  💾 Metadata saved

[6/16]

EXTRACTING: ජාතික තොටිල්ල –  ටිබෙට් ජාතික එස්. මහින්ද හිමි (Jathila Thotilla – S. Mahinda Thero)
Category: old-books
Tot

Extracting pages: 854it [00:00, ?it/s]

    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...


Extracting pages:   0%|          | 0/74 [00:00<?, ?it/s]

    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...

✓ Extraction complete
  Duration: 29.5s
  Content pages: 18/74
  Methods: 67 text-based, 7 OCR
  Avg OCR confidence: 0.812

[15/16]

EXTRACTING: අති උත්තම ආචාර්ය මන් භූරිදතත හිමි චරිතාපදානය (Most VenerableMan Buridaththa Thero’s Character)
Category: buddhist-characters
Total pages: 364


Extracting pages:   0%|          | 0/364 [00:00<?, ?it/s]

    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...


Extracting pages:   0%|          | 0/117 [00:00<?, ?it/s]

    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...
    Using Document AI (scanned page)...


## Delete extarcted files and restart extraction

In [ ]:
# # ============================================================================
# # EXTRACTION CLEANUP UTILITY
# # ============================================================================
# # Use this cell when you need to restart extraction from scratch
# # This will:
# #   1. Delete all extracted text files
# #   2. Remove 'extraction' field from metadata
# #   3. Optionally clean copyright_analysis to re-run filtering
# # ============================================================================

# import shutil

# def cleanup_extraction(delete_files=True,
#                        reset_extraction_metadata=True,
#                        reset_copyright_analysis=False,
#                        specific_books=None):
#     """
#     Clean up extraction data to restart from beginning.

#     Parameters:
#     -----------
#     delete_files : bool (default=True)
#         Delete all extracted text files and directories

#     reset_extraction_metadata : bool (default=True)
#         Remove 'extraction' field from metadata.json

#     reset_copyright_analysis : bool (default=False)
#         Remove 'copyright_analysis' field to re-run copyright filtering

#     specific_books : list of str (optional)
#         List of book names (Sinhala) to clean up.
#         If None, cleans ALL books.
#         Example: ["මෛත්‍රී බුද්ධ වංශය නොහොත් අනාගත වංශය"]

#     Returns:
#     --------
#     dict with cleanup statistics
#     """

#     print("\n" + "="*80)
#     print("EXTRACTION CLEANUP UTILITY")
#     print("="*80)

#     stats = {
#         'files_deleted': 0,
#         'directories_deleted': 0,
#         'books_cleaned': 0,
#         'extraction_removed': 0,
#         'copyright_removed': 0
#     }

#     # Load current metadata
#     metadata = load_json(METADATA_FILE, [])

#     if not metadata:
#         print("⚠️  No metadata found")
#         return stats

#     # ========================================================================
#     # STEP 1: Delete extracted files
#     # ========================================================================
#     if delete_files:
#         print("\n[1/3] Deleting extracted files...")

#         dirs_to_clean = [
#             EXTRACTED_DIR / "raw_text",
#             EXTRACTED_DIR / "filtered_text",
#             EXTRACTED_DIR / "low_confidence_pages",
#             EXTRACTED_DIR / "document_ai_cache",
#             EXTRACTED_DIR / "book_metadata",
#             EXTRACTED_DIR / "full_books"
#         ]

#         if specific_books:
#             # Clean only specific books
#             print(f"      Targeting {len(specific_books)} specific book(s)...")

#             for book in metadata:
#                 book_name = book.get('book_name_si', '')
#                 if book_name not in specific_books:
#                     continue

#                 category = book.get('Category', 'unknown')
#                 safe_name = sanitize_filename(book_name)

#                 # Delete book's directories
#                 book_dirs = [
#                     RAW_TEXT_DIR / category / safe_name,
#                     FILTERED_TEXT_DIR / category / safe_name,
#                     LOW_CONF_DIR / category / safe_name,
#                     DOCAI_CACHE_DIR / category / safe_name,
#                     BOOK_METADATA_DIR / category / safe_name
#                 ]

#                 for dir_path in book_dirs:
#                     if dir_path.exists():
#                         # Count files
#                         files_count = len(list(dir_path.glob('*')))
#                         stats['files_deleted'] += files_count

#                         shutil.rmtree(dir_path)
#                         stats['directories_deleted'] += 1
#                         print(f"      ✓ Deleted: {dir_path.relative_to(EXTRACTED_DIR)} ({files_count} files)")

#         else:
#             # Clean ALL extraction directories
#             print("      Cleaning ALL extracted files...")

#             for dir_path in dirs_to_clean:
#                 if dir_path.exists():
#                     # Count subdirectories and files
#                     subdirs = list(dir_path.glob('*/*'))
#                     files = list(dir_path.glob('*/*/*'))

#                     stats['directories_deleted'] += len(subdirs)
#                     stats['files_deleted'] += len(files)

#                     shutil.rmtree(dir_path)
#                     print(f"      ✓ Deleted: {dir_path.name}/ ({len(subdirs)} categories, {len(files)} files)")

#                     # Recreate empty directory
#                     dir_path.mkdir(parents=True, exist_ok=True)
#                 else:
#                     # Create if doesn't exist
#                     dir_path.mkdir(parents=True, exist_ok=True)
#                     print(f"      ✓ Created: {dir_path.name}/")

#     # ========================================================================
#     # STEP 2: Clean extraction metadata
#     # ========================================================================
#     if reset_extraction_metadata:
#         print("\n[2/3] Removing 'extraction' from metadata...")

#         for book in metadata:
#             book_name = book.get('book_name_si', '')

#             # If specific books specified, only clean those
#             if specific_books and book_name not in specific_books:
#                 continue

#             if 'extraction' in book:
#                 del book['extraction']
#                 stats['extraction_removed'] += 1

#                 if specific_books:
#                     print(f"      ✓ Cleaned: {book_name}")

#         if not specific_books:
#             print(f"      ✓ Removed extraction data from {stats['extraction_removed']} books")

#     # ========================================================================
#     # STEP 3: Clean copyright analysis (optional)
#     # ========================================================================
#     if reset_copyright_analysis:
#         print("\n[3/3] Removing 'copyright_analysis' from metadata...")

#         for book in metadata:
#             book_name = book.get('book_name_si', '')

#             # If specific books specified, only clean those
#             if specific_books and book_name not in specific_books:
#                 continue

#             if 'copyright_analysis' in book:
#                 del book['copyright_analysis']
#                 stats['copyright_removed'] += 1

#                 if specific_books:
#                     print(f"      ✓ Cleaned: {book_name}")

#         if not specific_books:
#             print(f"      ✓ Removed copyright analysis from {stats['copyright_removed']} books")
#     else:
#         print("\n[3/3] Keeping copyright_analysis (skip)")

#     # ========================================================================
#     # SAVE CLEANED METADATA
#     # ========================================================================
#     save_json(metadata, METADATA_FILE)
#     print(f"\n✓ Metadata saved to: {METADATA_FILE.name}")

#     # Count books affected
#     if specific_books:
#         stats['books_cleaned'] = len(specific_books)
#     else:
#         stats['books_cleaned'] = len(metadata)

#     # ========================================================================
#     # SUMMARY
#     # ========================================================================
#     print("\n" + "="*80)
#     print("CLEANUP SUMMARY")
#     print("="*80)
#     print(f"Books cleaned:           {stats['books_cleaned']}")
#     print(f"Directories deleted:     {stats['directories_deleted']}")
#     print(f"Files deleted:           {stats['files_deleted']}")
#     print(f"Extraction data removed: {stats['extraction_removed']}")
#     print(f"Copyright data removed:  {stats['copyright_removed']}")
#     print("="*80)
#     print("✅ Cleanup complete - ready for fresh extraction!")
#     print("="*80 + "\n")

#     return stats


# # ============================================================================
# # USAGE EXAMPLES
# # ============================================================================

# # Example 1: Complete clean restart (most common)
# # cleanup_extraction()

# # Example 2: Only remove metadata, keep files
# # cleanup_extraction(delete_files=False)

# # Example 3: Clean specific books only
# # cleanup_extraction(specific_books=[
# #     "මෛත්‍රී බුද්ධ වංශය නොහොත් අනාගත වංශය",
# #     "මහා සතිපට්ඨාන සූත්‍ර වර්ණනාව"
# # ])

# # Example 4: Full reset including copyright analysis
# # cleanup_extraction(reset_copyright_analysis=True)

# # Example 5: Clean specific book and re-run copyright filter
# # cleanup_extraction(
# #     specific_books=["මෛත්‍රී බුද්ධ වංශය නොහොත් අනාගත වංශය"],
# #     reset_copyright_analysis=True
# # )


# print("✓ Cleanup utility loaded")
# print("  Usage: cleanup_extraction()")
# print("  See cell for more options")

✓ Cleanup utility loaded
  Usage: cleanup_extraction()
  See cell for more options


In [ ]:
# cleanup_extraction()

## 11. Generate Extraction Report

In [18]:
# Generate comprehensive report
print("\n" + "="*80)
print("EXTRACTION STATISTICS")
print("="*80)

total_books = len(metadata)
extracted_books = [b for b in metadata if 'extraction' in b]
included_count = len([b for b in metadata if b.get('copyright_analysis', {}).get('can_include_in_corpus', False)])
excluded_count = total_books - included_count

print(f"\nCopyright Analysis:")
print(f"  Total books in metadata: {total_books}")
print(f"  Included (public domain): {included_count}")
print(f"  Excluded (copyright protected): {excluded_count}")
print(f"  Inclusion rate: {included_count/total_books*100:.1f}%")

print(f"\nExtraction Status:")
print(f"  Books extracted: {len(extracted_books)}")
print(f"  Pending extraction: {included_count - len(extracted_books)}")

if extracted_books:
    total_pages = sum(b['extraction']['page_info']['total_pages'] for b in extracted_books)
    content_pages = sum(b['extraction']['page_classification']['content'] for b in extracted_books)
    text_based = sum(b['extraction']['page_info']['text_based_pages'] for b in extracted_books)
    ocr_pages = sum(b['extraction']['page_info']['ocr_pages'] for b in extracted_books)

    print(f"\nPage Statistics:")
    print(f"  Total pages processed: {total_pages:,}")
    print(f"  Content pages: {content_pages:,} ({content_pages/total_pages*100:.1f}%)")
    print(f"  Text-based pages: {text_based:,} ({text_based/total_pages*100:.1f}%)")
    print(f"  OCR pages: {ocr_pages:,} ({ocr_pages/total_pages*100:.1f}%)")

    # OCR confidence statistics
    books_with_ocr = [b for b in extracted_books if 'ocr_confidence' in b['extraction']['quality_metrics']]
    if books_with_ocr:
        avg_confidences = [b['extraction']['quality_metrics']['ocr_confidence']['avg_confidence']
                          for b in books_with_ocr]
        overall_avg = sum(avg_confidences) / len(avg_confidences)

        print(f"\nOCR Confidence:")
        print(f"  Books with OCR: {len(books_with_ocr)}")
        print(f"  Average confidence: {overall_avg:.3f}")
        print(f"  Min confidence: {min(avg_confidences):.3f}")
        print(f"  Max confidence: {max(avg_confidences):.3f}")

# Category breakdown
print(f"\nBooks by Category:")
category_stats = {}
for book in metadata:
    cat = book['Category']
    if cat not in category_stats:
        category_stats[cat] = {'total': 0, 'included': 0, 'extracted': 0}
    category_stats[cat]['total'] += 1
    if book.get('copyright_analysis', {}).get('can_include_in_corpus', False):
        category_stats[cat]['included'] += 1
    if 'extraction' in book:
        category_stats[cat]['extracted'] += 1

for cat, stats in sorted(category_stats.items()):
    print(f"  {cat}:")
    print(f"    Total: {stats['total']}, Included: {stats['included']}, Extracted: {stats['extracted']}")

print("\n" + "="*80)


EXTRACTION STATISTICS

Copyright Analysis:
  Total books in metadata: 83
  Included (public domain): 16
  Excluded (copyright protected): 67
  Inclusion rate: 19.3%

Extraction Status:
  Books extracted: 16
  Pending extraction: 0

Page Statistics:
  Total pages processed: 7,523
  Content pages: 5,564 (74.0%)
  Text-based pages: 423 (5.6%)
  OCR pages: 5,565 (74.0%)

OCR Confidence:
  Books with OCR: 15
  Average confidence: 0.907
  Min confidence: 0.198
  Max confidence: 0.982

Books by Category:
  books-related-to-the-tipitaka:
    Total: 9, Included: 5, Extracted: 5
  buddhist-characters:
    Total: 4, Included: 3, Extracted: 3
  kukulpane-sudassi-himi:
    Total: 10, Included: 0, Extracted: 0
  mr-amaradasa-rathanapala:
    Total: 4, Included: 0, Extracted: 0
  old-books:
    Total: 10, Included: 8, Extracted: 8
  thripitaka-books:
    Total: 10, Included: 0, Extracted: 0
  ven-balangoda-ananda-maithriya-thero-books:
    Total: 1, Included: 0, Extracted: 0
  ven-galigamuwe-gnanade

## 12. Save Final Report

In [19]:
# Create comprehensive report
report = {
    'generated_at': datetime.now().isoformat(),
    'summary': {
        'total_books': total_books,
        'included_books': included_count,
        'excluded_books': excluded_count,
        'extracted_books': len(extracted_books),
        'inclusion_rate_percent': round(included_count/total_books*100, 2) if total_books > 0 else 0
    },
    'copyright_analysis': {
        'exclusion_reasons': exclusion_reasons,
        'policy': '70-year rule from author death (Sri Lankan IP Act No. 36 of 2003)',
        'cutoff_year': 1954
    },
    'extraction_statistics': {},
    'category_breakdown': category_stats
}

if extracted_books:
    report['extraction_statistics'] = {
        'total_pages': total_pages,
        'content_pages': content_pages,
        'text_based_pages': text_based,
        'ocr_pages': ocr_pages
    }

    if books_with_ocr:
        report['extraction_statistics']['ocr_confidence'] = {
            'books_with_ocr': len(books_with_ocr),
            'average_confidence': round(overall_avg, 3),
            'min_confidence': round(min(avg_confidences), 3),
            'max_confidence': round(max(avg_confidences), 3)
        }

# Save report
report_file = METADATA_DIR / 'extraction_report.json'
save_json(report, report_file)

print(f"\n✓ Report saved to: {report_file}")
print(f"✓ Updated metadata saved to: {METADATA_FILE}")
print("\n🎉 Extraction pipeline complete!")


✓ Report saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/extraction_report.json
✓ Updated metadata saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/metadata/metadata.json

🎉 Extraction pipeline complete!
